In [1]:
import glob
import cv2
import numpy as np

In [ ]:
# --- 1. CONFIGURATION ---
CHECKERBOARD = (6, 8)  # Inner corners of your checkerboard (columns, rows)
SQUARE_SIZE = 0.024  # Size of a single square side in meters (e.g., 25mm)
IMAGE_DIR = "dataset/"  # Directory containing saved stereo pairs

# Prepare 3D object points (e.g., (0,0,0), (1,0,0), (2,0,0) ...), scaled by square size
objp = np.zeros((CHECKERBOARD[0] * CHECKERBOARD[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:CHECKERBOARD[0], 0:CHECKERBOARD[1]].T.reshape(-1, 2) * SQUARE_SIZE

objpoints = []  # 3D points in real world space
imgpoints_l = []  # 2D points in left image plane
imgpoints_r = []  # 2D points in right image plane

# --- 2. LOAD AND PROCESS CALIBRATION IMAGES ---
# Expects naming convention: left_01.png / right_01.png, left_02.png / right_02.png, etc.
images_l = sorted(glob.glob(IMAGE_DIR + "*_left.jpg"))
images_r = sorted(glob.glob(IMAGE_DIR + "*_right.jpg"))

print(f"Found {len(images_l)} left images and {len(images_r)} right images.")

img_shape = None

for img_l_path, img_r_path in zip(images_l, images_r):
    img_l = cv2.imread(img_l_path)
    img_r = cv2.imread(img_r_path)

    gray_l = cv2.cvtColor(img_l, cv2.COLOR_BGR2GRAY)
    gray_r = cv2.cvtColor(img_r, cv2.COLOR_BGR2GRAY)
    gray_l = cv2.rotate(gray_l, cv2.ROTATE_90_COUNTERCLOCKWISE)
    gray_r = cv2.rotate(gray_r, cv2.ROTATE_90_COUNTERCLOCKWISE)

    if img_shape is None:
        img_shape = gray_l.shape[::-1]  # (width, height)

    # Find the chess board corners
    ret_l, corners_l = cv2.findChessboardCorners(gray_l, CHECKERBOARD, None)
    ret_r, corners_r = cv2.findChessboardCorners(gray_r, CHECKERBOARD, None)

    if ret_l and ret_r:
        objpoints.append(objp)

        # Refine corner locations for sub-pixel accuracy
        criteria = (
            cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER,
            30,
            0.001,
        )
        corners_l_sub = cv2.cornerSubPix(gray_l, corners_l, (11, 11), (-1, -1), criteria)
        corners_r_sub = cv2.cornerSubPix(gray_r, corners_r, (11, 11), (-1, -1), criteria)

        imgpoints_l.append(corners_l_sub)
        imgpoints_r.append(corners_r_sub)

print(f"Successfully extracted corners from {len(objpoints)} image pairs.")

# --- 3. PERFORM STEREO CALIBRATION ---
print("Running individual and stereo calibration...")

# Initialize intrinsic matrices for stable optimization convergence
mtx_l = np.eye(3, dtype=np.float32)
mtx_r = np.eye(3, dtype=np.float32)
dist_l = np.zeros((5, 1), dtype=np.float32)
dist_r = np.zeros((5, 1), dtype=np.float32)

flags = cv2.CALIB_FIX_INTRINSIC  # Or let it optimize intrinsics by removing flag

ret, mtx_l, dist_l, mtx_r, dist_r, R, T, E, F = cv2.stereoCalibrate(
    objpoints,
    imgpoints_l,
    imgpoints_r,
    mtx_l,
    dist_l,
    mtx_r,
    dist_r,
    img_shape,
    flags=0,
)

print(f"Stereo Calibration completed with RMS error: {ret:.4f}")

# --- 4. COMPUTE RECTIFICATION TRANSFORMS ---
R1, R2, P1, P2, Q, roi1, roi2 = cv2.stereoRectify(
    mtx_l, dist_l, mtx_r, dist_r, img_shape, R, T, alpha=0
)

# Generate lookup maps for fast remapping during live streaming
map1_l, map2_l = cv2.initUndistortRectifyMap(
    mtx_l, dist_l, R1, P1, img_shape, cv2.CV_16SC2
)
map1_r, map2_r = cv2.initUndistortRectifyMap(
    mtx_r, dist_r, R2, P2, img_shape, cv2.CV_16SC2
)

# --- 5. SAVE CALIBRATION TO DISK ---
output_filename = "stereo_calibration_params.npz"
np.savez(
    output_filename,
    mtx_l=mtx_l,
    dist_l=dist_l,
    mtx_r=mtx_r,
    dist_r=dist_r,
    R=R,
    T=T,
    R1=R1,
    R2=R2,
    P1=P1,
    P2=P2,
    Q=Q,
    map1_l=map1_l,
    map2_l=map2_l,
    map1_r=map1_r,
    map2_r=map2_r,
)

print(f"Calibration parameters successfully saved to {output_filename}!")

Found 21 left images and 21 right images.
Successfully extracted corners from 21 image pairs.
Running individual and stereo calibration...
Stereo Calibration completed with RMS error: 0.3931
Calibration parameters successfully saved to stereo_calibration_params.npz!


: 